# 05 Evaluation Report — понятный отчёт по текущему прогону

Эта тетрадка ничего нового не обучает и не считает с нуля тяжёлые модели. Она собирает результаты предыдущих тетрадок в один читаемый отчёт.

Её удобно открывать, когда нужно быстро понять:

- разметка вообще валидная или нет;
- сколько у нас точных дублей, разных фасовок и разных товаров;
- какой простой метод сейчас лучше;
- где методы ошибаются опасно;
- что происходит при сборке групп.


## Как читать этот отчёт

Не надо смотреть только на одну итоговую цифру. Для нашей задачи важнее баланс ошибок.

Особенно важны три вещи:

1. `exact_duplicate_precision` — насколько безопасны точные склейки.
2. `false_merge_rate` — как часто разные товары ошибочно связываются.
3. Примеры ошибок — они показывают, почему метод ошибается и что улучшать дальше.

Если `macro_f1` выглядит терпимо, но false merges много, метод всё равно нельзя считать готовым для финальной склейки SKU.


## Блок кода 1. Подготовка окружения

Эта ячейка подключает библиотеки, находит корень проекта и настраивает вывод таблиц.

Здесь нет бизнес-логики: это техническая подготовка, чтобы следующие ячейки могли читать CSV и красиво показывать таблицы.


In [1]:
from __future__ import annotations

from pathlib import Path
import sys

from IPython.display import Markdown, display
import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

pd.set_option("display.max_columns", 80)
pd.set_option("display.max_colwidth", 180)


## Блок кода 2. Загрузка всех готовых результатов

Эта ячейка читает файлы, созданные предыдущими этапами:

- разметку из `labeling_sauces.csv`;
- таблицу качества методов из `matching_summary_sauces.csv`;
- опасные ошибки из `matching_false_merges_sauces.csv`;
- результаты графовой сборки из `clustering_pair_eval_sauces.csv` и `clustering_components_sauces.csv`.

Если здесь ошибка про отсутствующий файл, значит сначала надо выполнить `03` и `04`.


In [2]:
DATA_DIR = PROJECT_ROOT / "research" / "dedup" / "data"
LABELING_PATH = DATA_DIR / "labeling_sauces.csv"
MATCHING_SUMMARY_PATH = DATA_DIR / "matching_summary_sauces.csv"
MATCHING_FALSE_MERGES_PATH = DATA_DIR / "matching_false_merges_sauces.csv"
CLUSTERING_PAIR_EVAL_PATH = DATA_DIR / "clustering_pair_eval_sauces.csv"
CLUSTERING_COMPONENTS_PATH = DATA_DIR / "clustering_components_sauces.csv"

required_paths = [
    LABELING_PATH,
    MATCHING_SUMMARY_PATH,
    MATCHING_FALSE_MERGES_PATH,
    CLUSTERING_PAIR_EVAL_PATH,
    CLUSTERING_COMPONENTS_PATH,
]
missing = [path for path in required_paths if not path.exists()]
if missing:
    raise FileNotFoundError("Run notebooks 03 and 04 first. Missing: " + ", ".join(str(path) for path in missing))

labels = pd.read_csv(LABELING_PATH)
matching_summary = pd.read_csv(MATCHING_SUMMARY_PATH)
false_merges = pd.read_csv(MATCHING_FALSE_MERGES_PATH)
pair_eval = pd.read_csv(CLUSTERING_PAIR_EVAL_PATH)
components = pd.read_csv(CLUSTERING_COMPONENTS_PATH)

print(f"Loaded labels: {len(labels)} rows")
print(f"Loaded matching summary: {len(matching_summary)} rows")
print(f"Loaded clustering pair eval: {len(pair_eval)} rows")


Loaded labels: 400 rows
Loaded matching summary: 6 rows
Loaded clustering pair eval: 379 rows


## Блок кода 3. Проверка разметки

Эта ячейка отвечает на вопрос: можно ли вообще доверять CSV с ручной разметкой как входу для метрик.

Смотри:

- `empty_labels` — пустые метки, их быть не должно;
- `invalid_labels` — неизвестные метки, их тоже быть не должно;
- распределение классов — сколько пар каждого типа;
- разрез по маркетплейсам — хватает ли межмаркетплейсных пар.

Если разметка грязная, любые метрики ниже будут сомнительными.


In [ ]:
allowed_labels = ["exact_duplicate", "different_product", "uncertain"]
label_values = labels["label"].fillna("").astype(str).str.strip()
label_distribution = label_values.value_counts().rename_axis("label").reset_index(name="pairs")
label_distribution["share"] = label_distribution["pairs"] / len(labels)
invalid_labels = sorted(set(label_values) - set(allowed_labels) - {""})
empty_labels = int(label_values.eq("").sum())

qa_summary = pd.DataFrame([
    {"check": "rows", "value": len(labels)},
    {"check": "empty_labels", "value": empty_labels},
    {"check": "invalid_labels", "value": len(invalid_labels)},
    {"check": "uncertain_pairs", "value": int(label_values.eq("uncertain").sum())},
])

display(qa_summary)
display(label_distribution)
if invalid_labels:
    display(Markdown("Invalid labels: " + ", ".join(invalid_labels)))

if "is_cross_marketplace_pair" in labels.columns:
    display(pd.crosstab(labels["is_cross_marketplace_pair"], label_values))


## Блок кода 4. Поиск подозрительных мест в разметке

Эта ячейка не исправляет разметку автоматически. Она только подсвечивает пары, которые стоит проверить глазами.

Например:

- legacy `same_product_different_pack` labels, которые нужно заменить на `exact_duplicate` перед бинарными метриками;
- `exact_duplicate`, где pack signature отличается: это уже не ошибка label, а ожидаемый вход для pack-правил;
- `different_product`, но модельный поиск нашёл очень высокое сходство и фасовка совпадает.

Такие строки не обязательно ошибки. Это список для ручного контроля качества.


In [ ]:
work = labels.copy()
for column in ["unit_amount_a", "unit_amount_b", "multipack_count_a", "multipack_count_b"]:
    if column in work.columns:
        work[column] = pd.to_numeric(work[column], errors="coerce")

same_unit = (work["unit_amount_a"] - work["unit_amount_b"]).abs().le(0.02)
same_pack = (work["multipack_count_a"] - work["multipack_count_b"]).abs().le(0.25)
pack_signature_same = same_unit & same_pack
label_norm = label_values

sanity_rows = pd.DataFrame([
    {"check": "legacy_same_product_different_pack_labels", "pairs": int(label_norm.eq("same_product_different_pack").sum())},
    {"check": "exact_duplicate_pack_signature_differs", "pairs": int(((label_norm == "exact_duplicate") & ~pack_signature_same).sum())},
    {"check": "different_product_same_pack_high_embedding", "pairs": int(((label_norm == "different_product") & pack_signature_same & work["embedding_similarity_score"].ge(0.95)).sum())},
])
display(sanity_rows)

display(Markdown("Sanity rows above are not automatic errors. They mark pairs worth revisiting if final metrics look unstable."))


## Блок кода 5. Сводная таблица качества методов

Эта ячейка показывает главную таблицу сравнения.

Как читать:

- `mode=default_full_gold_set_sanity` — быстрый обзор на всём размеченном наборе. Полезно, но не финальная честная оценка.
- `mode=calibrated`, `eval_split=dev` — часть, где подбирался порог.
- `mode=calibrated`, `eval_split=test` — самая важная строка для отчёта.

Смотри не только `macro_f1`, но и `exact_duplicate_precision` вместе с `false_merge_rate`.


In [5]:
display(matching_summary.sort_values(["mode", "eval_split", "macro_f1"], ascending=[True, True, False]))

test_summary = matching_summary[(matching_summary["mode"].eq("calibrated")) & (matching_summary["eval_split"].eq("test"))]
if test_summary.empty:
    selected_method = None
    display(Markdown("No calibrated held-out test summary found."))
else:
    selected = test_summary.sort_values("macro_f1", ascending=False).iloc[0]
    selected_method = selected["method"]
    display(Markdown(
        f"Selected current baseline: **{selected_method}** with held-out macro-F1 **{selected['macro_f1']:.3f}** "
        f"and false-merge rate **{selected['false_merge_rate']:.1%}**."
    ))


,method,mode,eval_split,threshold_high,pairs,macro_precision,macro_recall,macro_f1,exact_duplicate_precision,false_merge_count,false_merge_rate
2,rule_based_fuzzy,calibrated,dev,0.86,228,0.627863,0.626445,0.618742,0.517241,29,0.127193
4,bi_encoder_zero_shot,calibrated,dev,1.00,228,0.758277,0.545412,0.511972,1.000000,24,0.105263
3,rule_based_fuzzy,calibrated,test,0.86,151,0.619812,0.636099,0.606322,0.578947,28,0.185430
5,bi_encoder_zero_shot,calibrated,test,1.00,151,0.730000,0.539272,0.470139,1.000000,23,0.152318
0,rule_based_fuzzy,default_full_gold_set_sanity,all,0.82,379,0.591363,0.648863,0.606437,0.425532,89,0.234828
1,bi_encoder_zero_shot,default_full_gold_set_sanity,all,0.86,379,0.501348,0.582786,0.391180,0.262097,215,0.567282


Selected current baseline: **rule_based_fuzzy** with held-out macro-F1 **0.606** and false-merge rate **18.5%**.

## Блок кода 6. Конкретные опасные ошибки

Эта ячейка выводит пары, где выбранный метод сделал false merge.

Это значит: в ручной разметке пара была `different_product`, но метод решил связать её как `exact_duplicate`.

Эти примеры особенно полезны для следующего шага: они показывают, какие похожие товары метод путает.


In [ ]:
if selected_method is None:
    display(false_merges.head(0))
else:
    test_false_merges = false_merges[
        false_merges["method"].eq(selected_method) & false_merges["eval_split"].eq("test")
    ].copy()
    display(Markdown(f"Held-out false merges for **{selected_method}**: {len(test_false_merges)}"))
    columns = [
        "label",
        "predicted_label",
        "score",
        "threshold_high",
        "title_a",
        "title_b",
        "brand_a",
        "brand_b",
        "unit_amount_a",
        "unit_amount_b",
        "multipack_count_a",
        "multipack_count_b",
    ]
    display(test_false_merges[[column for column in columns if column in test_false_merges.columns]].head(15))


## Блок кода 7. Качество группировки

Эта ячейка берёт результат тетрадки `04` и кратко показывает качество графа.

`family` — проверка базовых товарных семей по binary positive labels.

`pack` — проверка конкретных фасовок после deterministic pack-правил.

`false_links` — лишние связи в графе. Они опасны, потому что могут склеить разные товары в одну группу.

`missed_links` — пропущенные связи. Они означают, что часть дублей осталась раздельно.


In [7]:
def _binary_link_report(frame: pd.DataFrame, *, true_col: str, pred_col: str, scope: str) -> dict[str, object]:
    true_link = frame[true_col].astype(bool)
    pred_link = frame[pred_col].astype(bool)
    tp = int((true_link & pred_link).sum())
    fp = int((~true_link & pred_link).sum())
    fn = int((true_link & ~pred_link).sum())
    tn = int((~true_link & ~pred_link).sum())
    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    return {
        "scope": scope,
        "pairs": len(frame),
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "false_links": fp,
        "missed_links": fn,
        "true_positive_links": tp,
        "true_negative_links": tn,
    }

heldout_pairs = pair_eval[pair_eval["eval_split"].eq("test")].copy()
cluster_report = pd.DataFrame([
    _binary_link_report(heldout_pairs, true_col="true_same_family", pred_col="pred_same_family", scope="family"),
    _binary_link_report(heldout_pairs, true_col="true_same_pack", pred_col="pred_same_pack", scope="pack"),
])
display(cluster_report)

component_summary = pd.DataFrame([
    {"graph": "pred_family", "components": components["pred_family_id"].nunique(), "multi_node_components": int((components.groupby("pred_family_id").size() > 1).sum())},
    {"graph": "pred_pack", "components": components["pred_pack_id"].nunique(), "multi_node_components": int((components.groupby("pred_pack_id").size() > 1).sum())},
    {"graph": "true_family_partial", "components": components["true_family_id"].nunique(), "multi_node_components": int((components.groupby("true_family_id").size() > 1).sum())},
    {"graph": "true_pack_partial", "components": components["true_pack_id"].nunique(), "multi_node_components": int((components.groupby("true_pack_id").size() > 1).sum())},
])
display(component_summary)


,scope,pairs,precision,recall,f1,false_links,missed_links,true_positive_links,true_negative_links
0,family,151,0.594203,0.694915,0.640625,28,18,41,64
1,pack,151,0.578947,0.379310,0.458333,8,18,11,114


,graph,components,multi_node_components
0,pred_family,560,144
1,pred_pack,668,48
2,true_family_partial,568,137
3,true_pack_partial,644,71


## Блок кода 8. Текущие выводы человеческим языком

Эта ячейка собирает короткий список выводов по текущему прогону.

Её удобно использовать как черновик для защиты или для следующего обсуждения: что уже доказали, что пока плохо, и какой следующий метод нужен.


In [8]:
conclusions = [
    "Gold-set заполнен: можно считать matching/clustering метрики.",
    "Rule-based calibrated baseline сейчас сильнее zero-shot bi-encoder по held-out macro-F1, но точность exact_duplicate всё ещё низкая для production merge.",
    "Zero-shot bi-encoder без reranker слишком агрессивен: при обычном threshold даёт много false merges, при threshold=1.0 почти теряет exact recall.",
    "Основной следующий ML-шаг: cross-encoder/reranker или более строгий fusion на hard negatives, особенно разные вкусы одного бренда и одинаковой фасовки.",
    "Перед финальной презентацией стоит вручную пересмотреть sanity-suspects из разметки, но технически CSV валиден.",
]

body = "## Current conclusions\n" + "\n".join(f"- {item}" for item in conclusions)
display(Markdown(body))


## Current conclusions
- Gold-set заполнен: можно считать matching/clustering метрики.
- Rule-based calibrated baseline сейчас сильнее zero-shot bi-encoder по held-out macro-F1, но точность exact_duplicate всё ещё низкая для production merge.
- Zero-shot bi-encoder без reranker слишком агрессивен: при обычном threshold даёт много false merges, при threshold=1.0 почти теряет exact recall.
- Основной следующий ML-шаг: cross-encoder/reranker или более строгий fusion на hard negatives, особенно разные вкусы одного бренда и одинаковой фасовки.
- Перед финальной презентацией стоит вручную пересмотреть sanity-suspects из разметки, но технически CSV валиден.

## Что делать после отчёта

Этот отчёт показывает, что первый baseline полезен как точка отсчёта, но ещё не годится как финальное решение.

Следующие практические шаги:

1. Глазами проверить подозрительные строки разметки.
2. Добавить более сильную проверку пары: cross-encoder или отдельный метод для спорных случаев.
3. Снова сравнить методы на том же `test`, не подгоняя пороги под него.
4. Только после выбора метода запускать его по полному списку кандидатов.
